# Final Project: Network Door Security System - (PYNQ #2)

Device Roles
PYNQ #1: Controls the alarm function of the door security system. Will emit a loud buzzing sound and flash a RGB board bright red when sound is detected with a sound sensor. This board will automatically be listening for the sound sensor board when the code is run.

PYNQ #2: Controls the sound sensor board. When a sound is detected it will send a signal to PYNQ board #2 and buzz the buzzer and flash the LED.

Wiring to this board:

Sensor Module Wiring to PYNQ PMODA
- A0 pin unconnected 
- (+) pin connected to 3.3V
- (-) pin connected to GND
- D0 pin connected to Pin 1

In [1]:
from multiprocessing import Process
from multiprocessing import Event
import threading
import time
from datetime import datetime
from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
import socket
import sys
import os

btns = base.btns_gpio
stop_program = Event()
button_pressed = True

In [2]:
%%microblaze base.PMODA
#include "gpio.h"

static int inited = 0;
static gpio sound_pin;

void init_sound_sensor()
{
    if (inited) return;

    sound_pin = gpio_open(1);          // PMODA pin 1
    gpio_set_direction(sound_pin, GPIO_IN);

    inited = 1;
}

// Return current digital value (0 or 1)
unsigned int read_sound()
{
    if (!inited) init_sound_sensor();
    return gpio_read(sound_pin);
}

In [3]:
import time

last_state = 0

while True:
    current_state = read_sound()

    # Rising edge detection (0 → 1)
    if current_state == 1 and last_state == 0:
        print("Sound detected!")

    last_state = current_state
    time.sleep(0.01)   # 10 ms delay to reduce CPU usage

Sound detected!



KeyboardInterrupt



In [5]:
def client(button_pressed):

    # Enter the other PYNQ board's IP Address
    # HOST = '192.168.XXX.XXX'
    # HOST = '192.168.0.204'
    # HOST = '192.168.230.2'
    HOST = '192.168.0.90'   
    
    # Loopback to IP Address of my PYNQ Board
    # HOST = '127.0.0.1'
    PORT = 50007
    
    s_client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s_client.connect((HOST, PORT))
    print("Client connected to server")

    # s_client.sendall(b'Hello, world')
    # data = s_client.recv(1024)
    # print('Received from server:', repr(data))
    
    # The board starts disarmed
    armed = False
    print("\nBUTTON OPTIONS:")
    print("Press Button 1 to arm security system...")
    print("Press Button 3 to disconnect...")

    
    while True:
        
        # Button 1 to arm security system
        if base.buttons[1].read() == 1:
            armed = True
            print("\n------------------------------------------------")
            print("Button 1 Pressed - Security system armed")
            print("Listening for intruders...")
            
            print("\nBUTTON OPTIONS:")
            print("Press Button 2 to disarm security system...")
            print("Press Button 3 to disconnect from server...")
            
            s_client.sendall(b'ARM')
            
            time.sleep(0.5)

        # Button 2 to disarm security system
        if base.buttons[2].read() == 1:
            armed = False
            print("\n------------------------------------------------")
            print("Button 2 Pressed - Security system disarmed")
            
            print("\nBUTTON OPTIONS:")
            print("Press Button 1 to arm security system...")
            print("Press Button 3 to disconnect from server...")
            
            s_client.sendall(b'DISARM')
            time.sleep(0.5)

        # Button 3 to disconnect from the alarm board
        if base.buttons[3].read() == 1:
            print("\n------------------------------------------------")
            print("Button 3 Pressed - Disconnecting from alarm board")
            s_client.sendall(b'DISCONNECT')
            break    
        
        if armed:
            sound = read_sound()
            if sound == 1:
                print("\n------------------------------------------------")
                print("UNAUTHORIZED ACCESS DETECTED")
                
                print("\nBUTTON OPTIONS:")
                print("Press Button 2 to disarm security system...")
                print("Press Button 3 to disconnect from server...")
                s_client.sendall(b'ALARM')
                time.sleep(1)
            
            time.sleep(0.1)
         

    # closes socket after the loop
    s_client.close()
    print("Client socket closed")

In [6]:
# Server process turns on upon code execution
button_pressed = True

# Button 0 Starts the Client
print("Press Button 0 to connect to alarm board...")
while base.buttons[0].read() == 0:
    time.sleep(0.1)

print("\n------------------------------------------------")
print("Button 0 Pressed")
print("starting client")

# Client process start
p1 = Process(target=client, args=(True,))
p1.start()

# Client Finishes
p1.join()

Press Button 0 to connect to alarm board...

------------------------------------------------
Button 0 Pressed
starting client
Client connected to server

BUTTON OPTIONS:
Press Button 1 to arm security system...
Press Button 3 to disconnect...

------------------------------------------------
Button 1 Pressed - Security system armed
Listening for intruders...

BUTTON OPTIONS:
Press Button 2 to disarm security system...
Press Button 3 to disconnect from server...

------------------------------------------------
UNAUTHORIZED ACCESS DETECTED

BUTTON OPTIONS:
Press Button 2 to disarm security system...
Press Button 3 to disconnect from server...

------------------------------------------------
Button 2 Pressed - Security system disarmed

BUTTON OPTIONS:
Press Button 1 to arm security system...
Press Button 3 to disconnect from server...

------------------------------------------------
Button 1 Pressed - Security system armed
Listening for intruders...

BUTTON OPTIONS:
Press Button 2 to 